<a href="https://colab.research.google.com/github/UdayLab/PAMI/blob/main/notebooks/fuzzyFrequentPattern/basic/MFFIMiner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finding Multiple Fuzzy Frequent Patterns using MFFIMiner

MFFIMiner retains every frequent region of an item. For example, `milk.Low` and `milk.High` can appear in separate patterns. A single pattern cannot contain both regions of `milk`.

This tutorial covers mining a local file, reading the patterns, saving results, using a DataFrame, comparing minimum support thresholds, and mining a dataset from a URL.

# Prerequisites

Use a PAMI installation that includes `MFFIMiner`. To install a local source checkout, run this command from the repository root:

```console
python3 -m pip install -e .
```

In Google Colab, install the source version in a code cell:

```python
%pip install "git+https://github.com/UdayLab/PAMI.git"
```

MFFIMiner runs on the CPU and does not require CUDA.

## Part 1: Mining a local fuzzy database

### Step 1: Import the libraries

In [ ]:
from pathlib import Path
import pandas as pd
from PAMI.fuzzyFrequentPattern.basic import MFFIMiner as alg

### Step 2: Create a sample fuzzy database

Each row uses `items:memberships`. The items and their memberships use the same separator and occur in the same order. This example uses spaces; the default separator in PAMI is a tab.

The input must already contain fuzzy memberships. Values must be finite numbers between 0 and 1, inclusive. Fuzzy item labels must be unique within a transaction. The final dot in a label separates the base item from its region.

In [ ]:
database = """milk.Low milk.High bread.High:0.8 0.2 0.9
milk.Low milk.High bread.High:0.3 0.7 0.6
milk.Low milk.High bread.High:0.6 0.4 0.8
milk.Low milk.High bread.High:0.1 0.9 0.4
"""
inputFile = "fuzzyTransactions.txt"
Path(inputFile).write_text(database, encoding="utf-8")
print(database)

### Step 3: Specify the minimum support

For a pattern, its contribution in one transaction is the minimum membership of its items, or zero if an item is absent. Its fuzzy support is the sum of these contributions across all transactions.

For example, `milk.High bread.High` has support:

```text
min(0.2, 0.9) + min(0.7, 0.6) + min(0.4, 0.8) + min(0.9, 0.4) = 1.6
```

An integer `minSup=1` sets an absolute fuzzy support threshold of 1. A float `minSup=0.25` sets the threshold to `0.25 * number_of_transactions`, which is also 1 for this four-row database. A pattern whose support equals the threshold is included.

The string `"1"` is an absolute threshold. The string `"1.0"` is a proportion of 1.0, meaning a threshold equal to the number of transactions.

In [ ]:
minimumSupportCount = 1
separator = ' '

### Step 4: Execute MFFIMiner

In [ ]:
obj = alg.MFFIMiner(iFile=inputFile, minSup=minimumSupportCount, sep=separator)
obj.mine()

### Step 5: Read and interpret the patterns

`getPatterns()` returns a dictionary. Each key is a tuple of sorted fuzzy item labels, and its value is the fuzzy support. `getPatternsAsDataFrame()` returns the columns `Patterns` and `Support`. Item labels in the DataFrame are joined using the specified separator.

In [ ]:
patterns = obj.getPatterns()
print(patterns)
patternsDF = obj.getPatternsAsDataFrame()
patternsDF.sort_values("Patterns").reset_index(drop=True)

The sample produces five patterns:

| Pattern | Fuzzy support |
| --- | --- |
| bread.High | 2.7 |
| milk.High | 2.2 |
| milk.Low | 1.8 |
| bread.High milk.High | 1.6 |
| bread.High milk.Low | 1.8 |

Both regions of `milk` are frequent. `milk.Low milk.High` is excluded because it contains two regions of the same base item. Support values may display additional decimal digits because they are stored as floating-point numbers.

### Step 6: Save the patterns

The output file contains one pattern per line in the format `fuzzyItem<sep>fuzzyItem:support`. The separator is the same one supplied to the miner.

In [ ]:
outputFile = "mffiPatterns.txt"
obj.save(outputFile)
print(Path(outputFile).read_text(encoding="utf-8"))

### Step 7: Retrieve runtime and memory measurements

Runtime is measured in seconds. RSS and USS are process memory measurements in bytes, taken after mining. They are not peak memory measurements or the additional memory used only by MFFIMiner.

In [ ]:
print("Total number of patterns:", len(patterns))
print("Runtime in seconds:", obj.getRuntime())
print("Memory RSS in bytes:", obj.getMemoryRSS())
print("Memory USS in bytes:", obj.getMemoryUSS())

`printResults()` prints these measurements together.

In [ ]:
obj.printResults()

## Part 2: Mining a DataFrame

The DataFrame must contain `Transactions` and `fuzzyValues` columns. Each row contains an item list and a membership list of equal length. Cells may also contain strings using the selected separator.

This example uses the same four transactions and a proportion threshold of 0.25.

In [ ]:
transactions = pd.DataFrame({
    'Transactions': [
        ['milk.Low', 'milk.High', 'bread.High'],
        ['milk.Low', 'milk.High', 'bread.High'],
        ['milk.Low', 'milk.High', 'bread.High'],
        ['milk.Low', 'milk.High', 'bread.High']
    ],
    'fuzzyValues': [
        [0.8, 0.2, 0.9],
        [0.3, 0.7, 0.6],
        [0.6, 0.4, 0.8],
        [0.1, 0.9, 0.4]
    ]
})

dataFrameObj = alg.MFFIMiner(transactions, minSup=0.25, sep=separator)
dataFrameObj.mine()
dataFrameObj.getPatternsAsDataFrame().sort_values('Patterns').reset_index(drop=True)

## Part 3: Comparing minimum support thresholds

The following example mines the small local database at three absolute thresholds. Increasing minimum support reduces the set of eligible patterns. Very low thresholds on large databases can produce many patterns and increase runtime and memory use.

In [ ]:
results = []
for minSupCount in [1, 2, 3]:
    miner = alg.MFFIMiner(inputFile, minSup=minSupCount, sep=separator)
    miner.mine()
    results.append({
        'minSup': minSupCount,
        'patterns': len(miner.getPatterns()),
        'runtime': miner.getRuntime(),
        'memoryRSS': miner.getMemoryRSS(),
        'memoryUSS': miner.getMemoryUSS()
    })

pd.DataFrame(results)

The expected pattern counts are 5, 2, and 0 for thresholds 1, 2, and 3, respectively. Runtime and memory depend on the environment.

## Part 4: Mining a fuzzy database from a URL

MFFIMiner also accepts a URL that returns a fuzzy database in the same text format. The dataset below uses tabs, so the default separator is used. A threshold of 0.05 means `0.05 * number_of_transactions`. This section requires internet access.

In [ ]:
inputURL = 'https://u-aizu.ac.jp/~udayrage/datasets/fuzzyDatabases/Fuzzy_T10I4D100K.csv'
urlObj = alg.MFFIMiner(inputURL, minSup=0.05)
urlObj.mine()
urlObj.save('mffiPatternsFromURL.txt')
urlObj.printResults()

## Running MFFIMiner from a terminal

From the installed source checkout, run:

```console
python3 PAMI/fuzzyFrequentPattern/basic/MFFIMiner.py fuzzyTransactions.txt mffiPatterns.txt 1 " "
```

The arguments are the input file, output file, minimum support, and optional separator. Omit the separator for tab-separated data. Use an integer such as `1` for an absolute threshold, or a decimal such as `0.25` for a proportion.

Call `mine()` in Python programs; `startMine()` is deprecated.

## Input restrictions

- Input must already contain fuzzy memberships; MFFIMiner does not convert quantitative data.
- Each text row must contain exactly two fields: `items:memberships`. Do not include a total utility or total membership field.
- Item and membership lists must have equal lengths.
- Memberships must be finite values in `[0, 1]`.
- Empty fuzzy item labels and repeated fuzzy item labels in one transaction are rejected.
- Use consistent `item.Region` labels so regions of the same base item are recognized.

## Reference

Lin et al., [Efficient Mining of Multiple Fuzzy Frequent Itemsets](https://doi.org/10.1007/s40815-016-0246-1).